# Example 02 — Two-DOF Chain of Oscillators with Cubic Spring

Frequency-response analysis of a 2-DOF chain of oscillators with a cubic spring nonlinearity.

**Reference**: `matlab/NLvib/EXAMPLES/02_twoDOFoscillator_cubicSpring/`

**Metric**: `a_rms = sqrt(sum(Q(1:2:end,:).^2)) / sqrt(2)` — RMS of DOF 0 across all harmonics.

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.nonlinearities.elements import cubic_spring, polynomial_stiffness
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

In [ ]:
# --- System setup (matches MATLAB example exactly) ---
MASSES      = [1.0, 0.05]
STIFFNESSES = [1.0, 0.0453, 0.0]
DAMPINGS    = [0.002, 0.013, 0.0]
K3_DOF0     = 1.0
K3_INTER    = 0.0042
F_AMP       = 0.11
H           = 7
OMEGA_START = 0.8
OMEGA_END   = 1.4

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
system.add_nonlinear_element(cubic_spring(k3=K3_DOF0, dof_index=0))

# Inter-DOF cubic coupling: k3_inter * (x0 - x1)^3 and reaction on DOF 1
_exp   = np.array([[3,0],[2,1],[1,2],[0,3]], dtype=np.intp)
_coeff = np.array([K3_INTER, -3*K3_INTER, 3*K3_INTER, -K3_INTER])
system.add_nonlinear_element(polynomial_stiffness(_exp, _coeff, np.array([0,1], dtype=np.intp)))
system.add_nonlinear_element(polynomial_stiffness(_exp, _coeff, np.array([1,0], dtype=np.intp)))

n_dof   = system.n_dof
n_total = n_dof * (2*H + 1)
print(f'System: {system}')
print(f'n_dof={n_dof}, H={H}, n_total_HB={n_total}')

In [ ]:
# --- Run HB continuation ---
excitation = {'dof': 0, 'amplitude': F_AMP}

# Initial Newton solve at OMEGA_START
Q = np.zeros(n_total)
for _ in range(50):
    R, J = hb_residual(Q, OMEGA_START, system, H, excitation)
    if np.linalg.norm(R) < 1e-10:
        break
    Q += np.linalg.solve(J, -R)

opts = ContinuationOptions(
    verbose=False, ds_initial=0.02, ds_min=1e-5, ds_max=0.1,
    max_steps=800, max_newton_iter=25, newton_tol=1e-8,
    adapt_step=True, lambda_min=OMEGA_START-0.05, lambda_max=OMEGA_END+0.05,
)
result = ContinuationSolver().run(
    lambda x, lam: hb_residual(x, lam, system, H, excitation),
    Q, OMEGA_START, opts
)
print(f'Continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Compute a_rms and plot FRF ---
solutions = result.solutions
omega_arr = solutions[:, -1]
Q_all     = solutions[:, :-1]

# a_rms for each DOF: sqrt(sum of all harmonic coefficients^2) / sqrt(2)
Q_dof0    = Q_all.reshape(Q_all.shape[0], 2*H+1, n_dof)[:, :, 0]
Q_dof1    = Q_all.reshape(Q_all.shape[0], 2*H+1, n_dof)[:, :, 1]
a_rms_dof0 = np.sqrt(np.sum(Q_dof0**2, axis=1)) / np.sqrt(2)
a_rms_dof1 = np.sqrt(np.sum(Q_dof1**2, axis=1)) / np.sqrt(2)

mask = (omega_arr >= OMEGA_START) & (omega_arr <= OMEGA_END)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(omega_arr[mask], a_rms_dof0[mask], 'b-',  linewidth=2,   label='DOF 0')
ax.plot(omega_arr[mask], a_rms_dof1[mask], 'b--', linewidth=1.5, label='DOF 1')
ax.set_xlabel('Excitation frequency (rad/s)')
ax.set_ylabel('a_rms (DOF 0, all harmonics)')
ax.set_title('Example 02 — Two-DOF Cubic Spring: FRF (Python HB)')
ax.set_xlim(OMEGA_START, OMEGA_END)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()

output_dir = Path('..') / 'examples' / '02_two_dof_cubic' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'frf.png', dpi=150, bbox_inches='tight')
print('Saved frf.png')

In [ ]:
# --- Display FRF inline ---
fig

In [ ]:
# --- Summary ---
peak_idx_dof0  = int(np.argmax(a_rms_dof0[mask]))
peak_idx_dof1  = int(np.argmax(a_rms_dof1[mask]))
omega_masked   = omega_arr[mask]

print('=' * 60)
print('SUMMARY — Example 02: Two-DOF Cubic Spring')
print('=' * 60)
print(f'  H (harmonics)        : {H}')
print(f'  F_AMP                : {F_AMP}')
print(f'  Peak a_rms  DOF 0    : {a_rms_dof0[mask][peak_idx_dof0]:.6f}  '
      f'at omega = {omega_masked[peak_idx_dof0]:.4f} rad/s')
print(f'  Peak a_rms  DOF 1    : {a_rms_dof1[mask][peak_idx_dof1]:.6f}  '
      f'at omega = {omega_masked[peak_idx_dof1]:.4f} rad/s')
print('=' * 60)